# CheXpert Active Labeling Inference

Cleaned batch odds-ratio workflow for the paper example. This version estimates Cardiomegaly odds for `Age >= 40` versus `Age < 40`. Original exploratory notebooks are preserved in `archive/`.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {'Stance', 'Alphafold', 'CheXpert', 'BRCA'} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    binary_odds_ratio_truth,
    HUMAN_N_COL,
    EFFECTIVE_N_COL,
    run_odds_ratio_monte_carlo,
    summarize_monte_carlo,
)
from plotting import (
    make_monte_carlo_variance_table,
    plot_coverage,
    plot_coverage_comparison,
    plot_effective_sample_size,
    plot_effective_sample_size_multiplier,
    plot_finite_population_coverage,
    plot_intervals,
    plot_monte_carlo_variance,
    plot_monte_carlo_variance_components,
    save_monte_carlo_variance_table,
)


In [2]:
EXAMPLE_DIR = REPO_ROOT / "CheXpert"
DATA_DIR = REPO_ROOT / "Data" / "CheXpert"
PLOTS_DIR = EXAMPLE_DIR / "plots"
RESULTS_DIR = EXAMPLE_DIR / "results"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 614
ALPHA = 0.1
TAU = 0.5
FRACS_HUMAN = np.linspace(0.1, 0.2, 20)
NUM_TRIALS = 500


In [3]:
master_df = pd.read_csv(DATA_DIR / "chex_chexmodel_withpreds_AP_PA.csv")

OUTCOME_COL = "Cardiomegaly"
PRED_COL = "prob_Cardiomegaly"
AGE_COL = "Age"

analysis_df = master_df.copy()
analysis_df[OUTCOME_COL] = pd.to_numeric(analysis_df[OUTCOME_COL], errors="coerce")
analysis_df[AGE_COL] = pd.to_numeric(analysis_df[AGE_COL], errors="coerce")
analysis_df = analysis_df[
    analysis_df[OUTCOME_COL].isin([0, 1])
    & analysis_df[PRED_COL].notna()
    & analysis_df[AGE_COL].notna()
].copy()
analysis_df["age_ge_40"] = analysis_df[AGE_COL] >= 40

Y = analysis_df[OUTCOME_COL].astype(int).to_numpy()
Yhat = analysis_df[PRED_COL].astype(float).to_numpy()
age_ge_40 = analysis_df["age_ge_40"].to_numpy(dtype=bool)

Y0, Yhat0 = Y[~age_ge_40], Yhat[~age_ge_40]
Y1, Yhat1 = Y[age_ge_40], Yhat[age_ge_40]
true_odds_ratio, true_variance = binary_odds_ratio_truth(Y0, Y1)
mu0_pilot, mu1_pilot = float(np.mean(Yhat0)), float(np.mean(Yhat1))

pd.DataFrame(
    {
        "group": ["Age < 40", "Age >= 40"],
        "n": [len(Y0), len(Y1)],
        "outcome_mean": [Y0.mean(), Y1.mean()],
        "prediction_mean": [Yhat0.mean(), Yhat1.mean()],
    }
)


,group,n,outcome_mean,prediction_mean
0,Age < 40,4063,0.141521,0.218163
1,Age >= 40,15533,0.383506,0.510004


In [4]:
df = run_odds_ratio_monte_carlo(
    y0=Y0,
    yhat0=Yhat0,
    y1=Y1,
    yhat1=Yhat1,
    fracs_human=FRACS_HUMAN,
    alpha=ALPHA,
    num_trials=NUM_TRIALS,
    true_odds_ratio=true_odds_ratio,
    true_variance=true_variance,
    mu0_pilot=mu0_pilot,
    mu1_pilot=mu1_pilot,
    tau=TAU,
    seed=SEED,
    split_spline_budget_evenly=False,
    show_progress=True,
)

summary_df = summarize_monte_carlo(df)
mc_variance_table = make_monte_carlo_variance_table(df)

df.to_csv(RESULTS_DIR / "CheXpert_results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "CheXpert_monte_carlo_summary.csv", index=False)
mc_variance_table.to_csv(RESULTS_DIR / "CheXpert_monte_carlo_variance_components.csv", index=False)

mc_variance_table.head(12)


human budget:   0%|          | 0/20 [00:00<?, ?it/s]

/Users/ginniema/miniconda3/lib/python3.9/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


trials 0.100:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.105:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.111:   0%|          | 0/500 [00:00<?, ?it/s]

...

trials 0.184:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.189:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.195:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.200:   0%|          | 0/500 [00:00<?, ?it/s]

,$n_{\mathrm{human}}$,estimator,point_estimate_mean,point_estimate_variance,lb_mean,lb_variance,ub_mean,ub_variance,interval_width_mean,interval_width_variance,coverage,finite_population_coverage,finite_population_interval_width,finite_population_variance_inflation
0,1959,active,3.801164,0.159860,3.139700,0.094643,4.603229,0.278238,1.463529,0.060998,0.934,0.906,1.329285,1.213305
1,1959,active + tuning,3.835080,0.156357,3.176020,0.097958,4.631815,0.256843,1.455795,0.047897,0.932,0.912,1.318373,1.219558
2,1959,classical,3.843059,0.342517,2.986960,0.177917,4.945344,0.656126,1.958384,0.153894,0.906,0.878,1.853757,1.117533
3,1959,spline,3.800221,0.072838,3.284506,0.054592,4.397067,0.098585,1.112561,0.008538,0.958,0.920,0.933342,1.419496
4,1959,spline + tuning,3.814593,0.068573,3.304588,0.051840,4.403443,0.091898,1.098855,0.007489,0.970,0.926,0.915463,1.439506
5,1959,uniform,3.827558,0.234055,3.096375,0.115774,4.733337,0.467462,1.636962,0.127121,0.920,0.898,1.515456,1.167553
6,2062,active,3.811772,0.155852,3.161104,0.092713,4.597527,0.269412,1.436424,0.057744,0.934,0.912,1.298340,1.225134
7,2062,active + tuning,3.847628,0.159693,3.199342,0.100636,4.628107,0.259904,1.428765,0.046483,0.924,0.898,1.287371,1.231783
8,2062,classical,3.804982,0.270912,2.978626,0.143680,4.861209,0.508546,1.882583,0.114081,0.940,0.924,1.776972,1.124278
9,2062,spline,3.790285,0.077610,3.287292,0.058528,4.370386,0.104239,1.083094,0.008509,0.958,0.918,0.899758,1.447858


In [5]:
n_total = len(Y0) + len(Y1)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "CheXpert_effective_sample_size.pdf",
    n_total=n_total,
    error_bars="sd",
    show=False,
)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "CheXpert_effective_sample_size_no_error_bars.pdf",
    n_total=n_total,
    error_bars="none",
    show=False,
)
plot_effective_sample_size_multiplier(
    df,
    path=PLOTS_DIR / "CheXpert_effective_sample_size_multiplier.pdf",
    n_total=n_total,
    error_bars="sd",
    show=False,
)
plot_effective_sample_size_multiplier(
    df,
    path=PLOTS_DIR / "CheXpert_effective_sample_size_multiplier_no_error_bars.pdf",
    n_total=n_total,
    error_bars="none",
    show=False,
)
plot_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "CheXpert_coverage.pdf",
    n_total=n_total,
    show=False,
)
plot_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "CheXpert_coverage_finite_population_calibrated.pdf",
    n_total=n_total,
    show=False,
)
plot_coverage_comparison(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "CheXpert_coverage_comparison.pdf",
    n_total=n_total,
    title="CheXpert Cardiomegaly Coverage",
    show=False,
)
plot_monte_carlo_variance(
    df,
    path=PLOTS_DIR / "CheXpert_monte_carlo_variance.pdf",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance_components(
    df,
    path=PLOTS_DIR / "CheXpert_monte_carlo_variance_components.pdf",
    n_total=n_total,
    show=False,
)
save_monte_carlo_variance_table(
    df,
    path=PLOTS_DIR / "CheXpert_monte_carlo_variance_table.pdf",
    max_rows=18,
    show=False,
)
plot_intervals(
    df,
    true_value=true_odds_ratio,
    path=PLOTS_DIR / "CheXpert_intervals.pdf",
    estimand_label="odds ratio: Age >= 40 vs Age < 40",
    show=False,
)


(<Figure size 700x840 with 1 Axes>,
 <Axes: xlabel='odds ratio: Age >= 40 vs Age < 40'>)